In [9]:
import pandas as pd

train = pd.read_csv('farm_train.csv')
test  = pd.read_csv('farm_test.csv')

In [10]:
train.head()

,농업면적,연도,지역,비료사용량,비료잔여량,작물종류,토양유형,농약검출여부,등급
0,20079.652837,2004,대구,407.985516,146.290507,보리,점토,2,C
1,73858.643204,2012,울산,221.229692,1967.333638,밀,점토,0,B
2,65718.150861,2012,강원,370.967205,2253.522610,쌀,점토,0,B
3,37366.182902,2005,광주,274.128236,1487.535265,쌀,양토,0,B
4,81515.151289,2007,충남,213.410655,683.306745,쌀,양토,1,B


In [11]:
test.head()

,농업면적,연도,지역,비료사용량,비료잔여량,작물종류,토양유형,등급
0,43284.730127,2008,울산,138.767165,415.437269,보리,점토,B
1,97624.943779,2014,충북,373.560368,2809.120053,보리,양토,A
2,47115.257674,2023,대전,249.082661,1586.452521,쌀,점토,B
3,82817.237255,2012,충북,242.284660,142.494412,쌀,점토,C
4,73866.403857,2012,광주,166.180358,4365.543337,밀,점토,A


In [12]:
train.isnull().sum()

농업면적      0
연도        0
지역        0
비료사용량     0
비료잔여량     0
작물종류      0
토양유형      0
농약검출여부    0
등급        0
dtype: int64

In [13]:
test.isnull().sum()

농업면적     0
연도       0
지역       0
비료사용량    0
비료잔여량    0
작물종류     0
토양유형     0
등급       0
dtype: int64

In [ ]:
target = train.pop('농약검출여부')
train_len = train.shape[0]

combined = pd.concat([train, test])

In [15]:
combined = pd.get_dummies(combined)
train = combined[:train_len]
test = combined[train_len:]

In [17]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(train, target, test_size=0.2, random_state=0)

print(X_tr.shape, X_val.shape)

(3200, 28) (800, 28)


In [18]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=0)
rf.fit(X_tr, y_tr)
pred = rf.predict(X_val)

In [19]:
from sklearn.metrics import f1_score

f1_score(y_val, pred, average='macro')

0.8462889767237595

In [20]:
from lightgbm import LGBMClassifier

lg = LGBMClassifier(random_state=0)
lg.fit(X_tr, y_tr)
pred_2 = lg.predict(X_val)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000233 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 838
[LightGBM] [Info] Number of data points in the train set: 3200, number of used features: 28
[LightGBM] [Info] Start training from score -0.830256
[LightGBM] [Info] Start training from score -2.743030
[LightGBM] [Info] Start training from score -0.693772
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


In [21]:
from sklearn.metrics import f1_score

f1_score(y_val, pred_2, average='macro')

0.9100316620356779

In [22]:
pred_2 = lg.predict(test)

pd.DataFrame({ 'pred': pred_2 }).to_csv('result.csv', index=False)